# 01 — Setup Environment

> **Tujuan:** Setup Colab environment, install dependencies, verify V-JEPA 2 pretrained model bisa di-load

> **Estimated time:** ~20-30 menit

---

## Step 1: GPU Check

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


✅ Pastikan GPU aktif (T4 atau lebih). Kalau "Not detected", klik:
**Runtime → Change runtime type → T4 GPU**

---

## Step 2: Install Dependencies

In [2]:
# Core dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# HuggingFace transformers + timm for ViT
!pip install transformers timm

# Data handling
!pip install pandas numpy scikit-learn

# Visualization
!pip install matplotlib seaborn pillow

# Experiment tracking
!pip install wandb

# Kaggle API
!pip install kaggle

Looking in indexes: https://download.pytorch.org/whl/cu121


---

## Step 3: Verify PyTorch & CUDA

In [4]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Memory check
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


---

## Step 4: Clone / Check jepa repo

V-JEPA 2 pretrained weights dan inference code ada di `facebookresearch/jepa`.

In [5]:
# Clone the repo (if not already cloned)
!git clone https://github.com/facebookresearch/jepa.git /content/jepa 2>/dev/null || echo "Repo already cloned"

import sys
sys.path.insert(0, '/content/jepa')

Repo already cloned


## Step 5: Verify V-JEPA 2 on HuggingFace

> **TL;DR:** V-JEPA 2 sudah tersedia di HuggingFace! Cek:

```python
# Model ID yang kamu butuh:
MODEL_ID = "facebook/vjepa2-vitl-fpc64-256"  # ViT-Large ~300M params
```

Browse models di: https://huggingface.co/models?search=v-jepa-2

> Steps 4-5 tentang `facebookresearch/jepa` repo → **legacy** (V-JEPA 1 only).
> Kita pakai HuggingFace untuk V-JEPA 2 — lebih simpel.

In [6]:
import os

# Check repo structure
!ls /content/jepa/

# Check for model configs
!find /content/jepa -name "*.yaml" -o -name "*.json" | head -20

app		    configs	     evals    README.md		setup.py
CODE_OF_CONDUCT.md  CONTRIBUTING.md  LICENSE  requirements.txt	src
/content/jepa/configs/evals/vith16_inat.yaml
/content/jepa/configs/evals/vitl16_k400_16x8x3.yaml
/content/jepa/configs/evals/vith16_places.yaml
/content/jepa/configs/evals/vith16_384_places.yaml
/content/jepa/configs/evals/vitl16_in1k.yaml
/content/jepa/configs/evals/vith16_k400_16x8x3.yaml
/content/jepa/configs/evals/vith16_384_k400_16x8x3.yaml
/content/jepa/configs/evals/vitl16_ssv2_16x2x3.yaml
/content/jepa/configs/evals/vitl16_inat.yaml
/content/jepa/configs/evals/vith16_in1k.yaml
/content/jepa/configs/evals/vith16_ssv2_16x2x3.yaml
/content/jepa/configs/evals/vith16_384_inat.yaml
/content/jepa/configs/evals/vith16_384_in1k.yaml
/content/jepa/configs/evals/vith16_384_ssv2_16x2x3.yaml
/content/jepa/configs/evals/vitl16_places.yaml
/content/jepa/configs/pretrain/vitl16.yaml
/content/jepa/configs/pretrain/vith16.yaml
/content/jepa/configs/pretrain/vith16_384.yaml


## Step 6: Install V-JEPA 2 via HuggingFace

V-JEPA 2 resmi tersedia di HuggingFace! Model yang kamu butuh:

| Model ID | Params | Resolution |
|---|---|---|
| `facebook/vjepa2-vitl-fpc64-256` | **0.3B (ViT-Large)** ← ini | 256×256 |
| `facebook/vjepa2-vith-fpc64-256` | 0.7B (ViT-Huge) | 256×256 |
| `facebook/vjepa2-vitg-fpc64-256` | 1B (ViT-Giant) | 256×256 |

> `vitl` = ViT-Large (~300M params, sesuai research design kamu)
> `fpc64` = frames per clip 64, tapi untuk image classification kamu bisa treat sebagai single frame

In [9]:
# Install torchcodec for video/image decoding
!pip install torchcodec

---

## Step 7: Quick Inference Test (Verify Model Loads)

Setelah dapat pretrained weights, test apakah model bisa di-load dan give reasonable output shape.

In [10]:
import torch
from transformers import AutoVideoProcessor, AutoModel
from PIL import Image
import numpy as np

# ============================================================
# V-JEPA 2 — Model Loading & Inference Test
# ============================================================

MODEL_ID = "facebook/vjepa2-vitl-fpc64-256"

print(f"Loading model: {MODEL_ID}")
print("Downloading from HuggingFace (first time ~1.2GB, wait...)")

processor = AutoVideoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"\n✅ Model loaded successfully!")
print(f"   Device: {device}")
print(f"   Params: ~0.3B (ViT-Large)")

# ============================================================
# Test inference: single chest X-ray image
# For X-ray (single image), repeat image 16x to simulate video frames
# ============================================================

print("\nRunning inference test with dummy X-ray image...")

# Create a dummy grayscale-like image (simulating 256x256 X-ray)
dummy_image = Image.fromarray(
    np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)
)

# Process image → video format (repeat 16x for temporal dimension)
pixel_values = processor(
    dummy_image, return_tensors="pt"
)["pixel_values_videos"]  # [1, 16, 3, 256, 256]
pixel_values = pixel_values.repeat(1, 4, 1, 1, 1)  # Make it 64 frames (fpc64)

pixel_values = pixel_values.to(device)
print(f"   Input shape: {pixel_values.shape}  [B, T, C, H, W]")

# Extract features
with torch.no_grad():
    features = model.get_vision_features(pixel_values)

print(f"   Output feature shape: {features.shape}")
print(f"   Feature dim: {features.shape[-1]}")


# Memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(device) / 1e9
    print(f"   GPU memory used: {allocated:.2f} GB")

print("\n✅ V-JEPA 2 inference test PASSED!")

Loading model: facebook/vjepa2-vitl-fpc64-256


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]


✅ Model loaded successfully!
   Device: cuda
   Params: ~0.3B (ViT-Large)

Running inference test with dummy X-ray image...
   Input shape: torch.Size([1, 4, 3, 256, 256])  [B, T, C, H, W]
   Output feature shape: torch.Size([1, 512, 1024])
   Feature dim: 1024
   GPU memory used: 1.32 GB

✅ V-JEPA 2 inference test PASSED!


## Checklist — Environment Setup

- [x] GPU detected (T4, 16 GB)
- [x] PyTorch + CUDA working
- [x] transformers (latest from GitHub) + torchcodec installed
- [x] V-JEPA 2 loaded from HuggingFace (`facebook/vjepa2-vitl-fpc64-256`)
- [x] Model inference test passed
- [ ] Download Kaggle Chest X-Ray dataset (→ notebook 02)

---

## Next Step

**02_data_preparation.ipynb** — download dan split dataset Kaggle Chest X-Ray.

> ⚠️ Kamu butuh Kaggle API token untuk download dataset.
> Buat di: https://www.kaggle.com → Account → Create New API Token